<a href="https://colab.research.google.com/github/jayanthkorupolu2000-web/Final_Project/blob/main/Attention_Enhanced_Cnn_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/crack_dataset"

print("Train exists:", os.path.exists(BASE_PATH + "/train"))
print("Valid exists:", os.path.exists(BASE_PATH + "/valid"))
print("Test exists :", os.path.exists(BASE_PATH + "/test"))


Train exists: True
Valid exists: True
Test exists : True


In [ ]:
with open(BASE_PATH + "/train/_annotations.coco.json", "r") as f:
    coco = json.load(f)

img_id_to_name = {img["id"]: img["file_name"] for img in coco["images"]}

# group bboxes per image
img_to_bboxes = defaultdict(list)
for ann in coco["annotations"]:
    img_to_bboxes[ann["image_id"]].append(ann["bbox"])

print("Images:", len(img_to_bboxes))
print("Annotations:", len(coco["annotations"]))


Images: 735
Annotations: 4233


In [ ]:
image_ids = list(img_to_bboxes.keys())
random.shuffle(image_ids)

split_idx = int(0.8 * len(image_ids))
train_img_ids = set(image_ids[:split_idx])
val_img_ids   = set(image_ids[split_idx:])

print("Train images:", len(train_img_ids))
print("Val images  :", len(val_img_ids))


Train images: 588
Val images  : 147


In [ ]:
def train_generator():
    for img_id in train_img_ids:
        img_name = img_id_to_name[img_id]
        img_path = os.path.join(TRAIN_IMG_DIR, img_name)
        if not os.path.exists(img_path):
            continue

        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        bboxes = img_to_bboxes[img_id]

        # --- positive samples ---
        for x, y, w, h in bboxes:
            x, y, w, h = map(int, [x, y, w, h])
            if w * h < MIN_CRACK_AREA:
                continue
            yield np.array(img.crop((x, y, x + w, y + h))), 1

        # --- negative samples ---
        for _ in range(len(bboxes)):
            rx = random.randint(0, max(0, W - 64))
            ry = random.randint(0, max(0, H - 64))
            yield np.array(img.crop((rx, ry, rx + 64, ry + 64))), 0


def val_generator():
    for img_id in val_img_ids:
        img_name = img_id_to_name[img_id]
        img_path = os.path.join(TRAIN_IMG_DIR, img_name)
        if not os.path.exists(img_path):
            continue

        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        bboxes = img_to_bboxes[img_id]

        # positive
        for x, y, w, h in bboxes:
            x, y, w, h = map(int, [x, y, w, h])
            if w * h < MIN_CRACK_AREA:
                continue
            yield np.array(img.crop((x, y, x + w, y + h))), 1

        # negative
        for _ in range(len(bboxes)):
            rx = random.randint(0, max(0, W - 64))
            ry = random.randint(0, max(0, H - 64))
            yield np.array(img.crop((rx, ry, rx + 64, ry + 64))), 0


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

def preprocess_tf(img, label):
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

train_ds = tf.data.Dataset.from_generator(
    train_generator,
    output_signature=(
        tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
).map(preprocess_tf).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


val_ds = tf.data.Dataset.from_generator(
    val_generator,
    output_signature=(
        tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
).map(preprocess_tf).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Train / Val datasets ready (no leakage)")



Train / Val datasets ready (no leakage)


In [ ]:
import tensorflow as tf

# Compile model
attn_cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Checkpoint (best model)
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/attn_cnn_checkpoint.keras",
    save_best_only=True,
    monitor="val_loss"
)

# Train
history_attn = attn_cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[checkpoint_cb]
)

# Save final model
attn_cnn_model.save("/content/drive/MyDrive/attn_cnn_model.keras")

print("Model-2 (Attention-CNN) trained and saved")


Epoch 1/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 27s 51ms/step - accuracy: 0.4909 - loss: 5.1816 - val_accuracy: 0.5239 - val_loss: 0.6968
Epoch 2/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 17s 39ms/step - accuracy: 0.5149 - loss: 0.6950 - val_accuracy: 0.5112 - val_loss: 0.6910
Epoch 3/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 17s 38ms/step - accuracy: 0.5166 - loss: 0.6921 - val_accuracy: 0.5402 - val_loss: 0.6904
Epoch 4/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 20s 39ms/step - accuracy: 0.5178 - loss: 0.6918 - val_accuracy: 0.5239 - val_loss: 0.6889
Epoch 5/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 17s 39ms/step - accuracy: 0.5236 - loss: 0.6912 - val_accuracy: 0.5239 - val_loss: 0.6889
Epoch 6/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 20s 46ms/step - accuracy: 0.5427 - loss: 0.6875 - val_accuracy: 0.5747 - val_loss: 0.6807
Epoch 7/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 16s 38ms/step - accuracy: 0.5669 - loss: 0.6831 - val_accuracy: 0.5245 - val_loss: 0.6718
Epoch 8/20
401/401 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.5736 - loss: 0.6702 - 